In [12]:
%pip install SpeechRecognition

Note: you may need to restart the kernel to use updated packages.


In [13]:
%pip install python-dotenv SpeechRecognition gTTS pygame PyAudio

Note: you may need to restart the kernel to use updated packages.


In [4]:
import os
import time
from dotenv import load_dotenv
import speech_recognition as sr
from gtts import gTTS
import pygame
import requests

load_dotenv()
api_key = os.getenv("API_KEY")

def VoiceToText():
    recognizer = sr.Recognizer()
    with sr.Microphone() as source:
        print("Listening... Speak now!")
        try:
            # Calibrate for room noise
            recognizer.adjust_for_ambient_noise(source, duration=0.5)
            voice = recognizer.listen(source)
            print("Processing voice...")
            text = recognizer.recognize_google(voice)
            return text.lower()
        except:
            return None

def text_to_voice(text):
    pygame.mixer.init()
    # lang='en' for standard English output
    tts = gTTS(text=text, lang='en', slow=False)
    tts.save("welcome.mp3")
    pygame.mixer.music.load("welcome.mp3")
    pygame.mixer.music.play()
    while pygame.mixer.music.get_busy():
        time.sleep(0.1)
    pygame.mixer.music.unload()

def get_weather(city):

    url = "https://api.openweathermap.org/data/2.5/weather"

    params = {
        "q": city.strip(),
        "appid": api_key,
        "units": "metric"
    }

    try:
        response = requests.get(url, params=params, timeout=10)
        data = response.json()

        if response.status_code != 200:
            print("API Error:", data)
            return None

        description = data["weather"][0]["description"]
        temp = data["main"]["temp"]
        humidity = data["main"]["humidity"]

        return (
            f"The weather in {city} is {description}. "
            f"The temperature is {temp} degrees Celsius "
            f"and humidity is {humidity} percent."
        )

    except requests.exceptions.RequestException as e:
        print("Connection Error:", e)
        return None
        # Main Execution

city = VoiceToText()

print("City:", city)

if city:
    weather_text = get_weather(city)

    if weather_text is None:
        weather_text = "Sorry, city not found. Please try again."

    print("Weather Info:", weather_text)

    text_to_voice(weather_text)

else:
    print("Could not capture your voice. Please run again.")

Listening... Speak now!
Processing voice...
City: tanuku
Weather Info: The weather in tanuku is moderate rain. The temperature is 30.45 degrees Celsius and humidity is 67 percent.
